# Episode 10: Build a Customer Service System

Time to put it all together. You've learned the patterns — now let's build a real system.

In this episode, you'll build a complete customer service system with triage, billing, and tech support agents. I'll provide the architecture. You fill in the key pieces.

## Planning the system

Before writing code, let's plan the architecture. Multi-agent customer service isn't hypothetical — production systems handle 50K daily interactions with 58% faster resolution and 84% first-call resolution rates. A good system needs three things:

1. **Triage** — A front-line agent that classifies incoming queries
2. **Specialists** — Agents with domain expertise (billing, tech support, general)
3. **Control flow** — After a specialist responds, control returns to the user

Here's the routing diagram:

```
User → Triage → Billing Agent  (billing questions)
              → Tech Agent     (technical issues)
              → General Agent  (everything else)
```

This is a classic **DefaultPattern with handoffs** — the triage agent has explicit routing rules, and each specialist reverts back to the user when done.

## Step 1: Set up the foundation

We start with imports, LLM configuration, and customer context variables to track state across the conversation. Let's see this:

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group.patterns import DefaultPattern
from autogen.agentchat.group import (
    AgentTarget,
    RevertToUserTarget,
    OnCondition,
    StringLLMCondition,
    ContextVariables,
    TerminateTarget,
)

llm_config = LLMConfig({"model": "gpt-4o-mini"})

# Track customer state across the conversation
context = ContextVariables(
    data={
        "customer_name": "Alice",
        "issue_type": None,
        "resolved": False,
    }
)

## Step 2: Create the triage agent

Now that we have our foundation, let's build the front door. The triage agent reads the customer's message and routes to the right specialist. It never tries to solve the problem itself.

In [ ]:
triage_agent = ConversableAgent(
    name="triage_agent",
    system_message=(
        "You are a customer service triage agent. Your job is to greet the customer "
        "and route their query to the right specialist.\n\n"
        "Rules:\n"
        "- For billing, charges, invoices, or payment questions -> hand off to billing\n"
        "- For technical issues, bugs, errors, or setup help -> hand off to tech support\n"
        "- For anything else -> hand off to general support\n\n"
        "Never try to solve the problem yourself. Just classify and route."
    ),
    llm_config=llm_config,
        human_input_mode="NEVER",
)

## Step 3: Create specialist agents

This leads to the agents that actually solve problems. Each specialist has domain-specific instructions — billing knows about charges and refunds, tech knows about errors and troubleshooting. We also add a general agent as a catch-all.

Let's see this:

In [ ]:
billing_agent = ConversableAgent(
    name="billing_agent",
    system_message=(
        "You are a billing support specialist. You handle questions about charges, "
        "invoices, payment methods, refunds, and account billing.\n\n"
        "When helping a customer:\n"
        "1. Acknowledge their billing concern\n"
        "2. Use available tools to look up account information\n"
        "3. Explain charges or resolve billing issues clearly\n"
        "4. Provide a concise summary of the resolution"
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

tech_agent = ConversableAgent(
    name="tech_agent",
    system_message=(
        "You are a technical support specialist. You handle bugs, errors, setup issues, "
        "and technical troubleshooting.\n\n"
        "When helping a customer:\n"
        "1. Acknowledge the technical issue\n"
        "2. Use available tools to check system status\n"
        "3. Provide step-by-step troubleshooting guidance\n"
        "4. Confirm the issue is resolved or escalate if needed"
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

general_agent = ConversableAgent(
    name="general_agent",
    system_message=(
        "You are a general support agent. You handle questions that don't fall into "
        "billing or technical categories -- things like account settings, feature requests, "
        "product information, and general inquiries.\n\n"
        "Be helpful, friendly, and concise."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

## Step 4: Add tools

Now that you've seen the agents, let's give the specialists mock tools so they can "look things up." In a real system, these would call databases or APIs. Let's see this:

In [ ]:
def check_billing_status(account_id: str) -> str:
    """Check the billing status for a customer account."""
    accounts = {
        "ACC-001": {"balance": "$0.00", "status": "current", "last_payment": "2026-03-01"},
        "ACC-002": {"balance": "$45.99", "status": "overdue", "last_payment": "2026-01-15"},
    }
    account = accounts.get(account_id, None)
    if account:
        return f"Account {account_id}: Balance={account['balance']}, Status={account['status']}, Last Payment={account['last_payment']}"
    return f"Account {account_id} not found. Please verify the account ID."


def check_system_status() -> str:
    """Check the current system status for known outages or issues."""
    return (
        "System Status Report:\n"
        "- API: Operational\n"
        "- Dashboard: Operational\n"
        "- Payment Processing: Degraded (investigating)\n"
        "- Email Notifications: Operational"
    )


# Register tools: LLM agents suggest the call, user agent executes it
user = ConversableAgent(name="user", human_input_mode="NEVER")

billing_agent.register_for_llm()(check_billing_status)
user.register_for_execution()(check_billing_status)

tech_agent.register_for_llm()(check_system_status)
user.register_for_execution()(check_system_status)

## Step 5: Configure routing

Now we wire up the handoffs. The triage agent uses `OnCondition` with `StringLLMCondition` to route based on the content of the customer's message. Each specialist uses `set_after_work(RevertToUserTarget())` so control returns to the user after they respond.

This is where the architecture comes together. Let's see this:

In [ ]:
# Set up triage handoffs
triage_agent.handoffs = [
    OnCondition(
        target=AgentTarget(billing_agent),
        condition=StringLLMCondition(
            prompt="The customer has a billing, payment, invoice, or charges question."
        ),
    ),
    OnCondition(
        target=AgentTarget(tech_agent),
        condition=StringLLMCondition(
            prompt="The customer has a technical issue, bug, error, or setup problem."
        ),
    ),
    OnCondition(
        target=AgentTarget(general_agent),
        condition=StringLLMCondition(
            prompt="The customer has a general question that is not about billing or technical issues."
        ),
    ),
]

# After specialists respond, return control to the user
billing_agent.set_after_work(RevertToUserTarget())
tech_agent.set_after_work(RevertToUserTarget())
general_agent.set_after_work(RevertToUserTarget())

## Step 6: Wire it up and test

Now that you've seen all the pieces, let's create a `DefaultPattern` and run the system with three different test queries.

In [ ]:
pattern = DefaultPattern(
    initial_agent=triage_agent,
    agents=[triage_agent, billing_agent, tech_agent, general_agent],
    context_variables=context,
)

# Test 1: Billing query
print("=" * 60)
print("TEST 1: Billing Query")
print("=" * 60)
result, context_out, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="I was charged twice on my last invoice. My account is ACC-002.",
    max_rounds=6,
)

In [ ]:
# Test 2: Tech query
print("=" * 60)
print("TEST 2: Technical Query")
print("=" * 60)
result, context_out, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="The dashboard keeps showing a 500 error when I try to load my reports.",
    max_rounds=6,
)

In [ ]:
# Test 3: General query
print("=" * 60)
print("TEST 3: General Query")
print("=" * 60)
result, context_out, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="What features are included in the Pro plan?",
    max_rounds=6,
)

## Design rationale

**DefaultPattern vs. AutoPattern: trade-offs**

Customer support routing must be **predictable**. When a customer asks about billing, they should always reach the billing agent — not sometimes the tech agent because the LLM felt like it. DefaultPattern with explicit handoff conditions gives us that reliability.

The trade-off is flexibility. AutoPattern would work for open-ended conversations where you genuinely don't know which agent should speak next. But in support systems, misrouting a customer is worse than being slightly rigid.

**Why these specific conditions?**

Billing, technical, and general covers roughly 90% of real support queries. The `StringLLMCondition` prompts are written to be mutually exclusive — a query about "charges" clearly maps to billing, not tech. The general agent acts as a catch-all so no query falls through the cracks.

**Why `RevertToUserTarget()` instead of `TerminateTarget()`?**

In a real chat interface, you want the user to be able to ask follow-up questions. `RevertToUserTarget()` keeps the conversation open. Use `TerminateTarget()` only for batch processing where no follow-up is expected.

## Try it yourself

What happens if a customer wants to return something? Right now the system would route that to the general agent. Can you do better?

Try adding a 4th specialist: a **returns/refund agent** that handles return requests, refund processing, and exchange queries. You'll need to:

1. Create a `returns_agent` with an appropriate system message
2. Add a mock tool like `process_return(order_id)`
3. Add another `OnCondition` to the triage agent's handoffs
4. Set `after_work` to `RevertToUserTarget()`
5. Add it to the pattern's agent list
6. Test it with a message like "I'd like to return the headphones I ordered last week"

In [ ]:
# Your code here


## What's next?

Your customer service system routes and resolves — but what if you needed agents that collaborate on a longer task, passing work back and forth until it's right? That's exactly what a research pipeline does.